# Plan 003.2 — Content-level tags

A public, synthetic walkthrough of Archiver's implemented tag infrastructure.

Archiver tags **content identity**, not filenames. This matters because paths can be renamed, disappear, or duplicate the same bytes, while metadata about the content should remain stable.

## Audience, prerequisites, and learning goals

This tutorial is for developers and users who understand the catalog and want to attach descriptive metadata safely.

Prerequisites:

- Python 3.12+ and the project environment (`uv sync`)
- familiarity with catalog scans and SHA-256 content identity

By the end you will be able to:

1. tag content through a current path;
2. observe the same tag through duplicate paths;
3. record user or system provenance, including tool name and version;
4. query tagged content with bounded results;
5. understand why tags survive refreshes and disappearing paths;
6. use the equivalent CLI commands.

## Why tags belong to content

A pathname is an observation inside one scan. It is not the identity of the bytes. If two paths have the same SHA-256 digest, tagging either path should describe the shared content once.

Plan 003.2 also keeps provenance with every assertion:

- `kind`: `user` or `system`;
- producer name and version;
- optional stable method/configuration detail;
- assertion and retraction timestamps in the catalog.

Different tools or tool versions can coexist without silently overwriting each other.

## 1. Create a disposable catalog

All files are synthetic and live in a temporary directory.

In [ ]:
from __future__ import annotations

from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory

from archiver import Catalog, TagProvenance

workspace = TemporaryDirectory()
base = Path(workspace.name)
root = base / "library"
root.mkdir()

first = root / "photos" / "first.bin"
second = root / "copies" / "second.bin"
unique = root / "notes" / "unique.txt"
for path in (first, second, unique):
    path.parent.mkdir(parents=True, exist_ok=True)

first.write_bytes(b"shared example bytes")
second.write_bytes(b"shared example bytes")
unique.write_text("different content", encoding="utf-8")
source_state = {
    path: (path.read_bytes(), path.stat().st_mtime_ns, path.stat().st_mode)
    for path in (first, second, unique)
}

catalog = Catalog.create(base / "catalog.sqlite")
summary = catalog.scan_directory(root)
print(f"Observed {summary.files_observed} files and {summary.distinct_content_count} content identities")
print(f"Schema version: {catalog.schema_version}")

## 2. Apply a user tag through one duplicate

The path is only a lookup mechanism. Archiver resolves it to a `ContentId`, then stores the assertion against that content record.

In [ ]:
user_source = TagProvenance(
    kind="user",
    source_name="tutorial",
    source_version="1.0",
)

created = catalog.add_tag_for_path(
    root,
    PurePosixPath("photos/first.bin"),
    "favorite",
    user_source,
)
first_tags = catalog.tags_for_path(root, PurePosixPath("photos/first.bin"))
second_tags = catalog.tags_for_path(root, PurePosixPath("copies/second.bin"))

assert created is True
assert first_tags == second_tags
print(f"First path tags:  {[item.tag for item in first_tags]}")
print(f"Second path tags: {[item.tag for item in second_tags]}")
print(f"Shared digest: {first_tags[0].content_id.digest[:16]}…")

### Idempotency

Repeating the same content/tag/exact-provenance assertion does not create another active row. The operation reports that nothing changed.

In [ ]:
created_again = catalog.add_tag_for_path(
    root,
    PurePosixPath("copies/second.bin"),
    "favorite",
    user_source,
)
assert created_again is False
assert {
    path: (path.read_bytes(), path.stat().st_mtime_ns, path.stat().st_mode)
    for path in (first, second, unique)
} == source_state
print("Repeated exact assertion changed catalog state:", created_again)
print("Tagging left source bytes and filesystem metadata unchanged")

## 3. Preserve system producer identity

The example below simulates two versions of a deterministic classifier. Automatic classification is **not** part of Plan 003.2; these assertions demonstrate the infrastructure future tools will use.

In [ ]:
content_id = catalog.content_for_path(root, PurePosixPath("photos/first.bin"))
classifier_v1 = TagProvenance("system", "demo-classifier", "1.0", "rules=alpha")
classifier_v2 = TagProvenance("system", "demo-classifier", "2.0", "rules=beta")

catalog.add_content_tag(content_id, "binary", classifier_v1)
catalog.add_content_tag(content_id, "binary", classifier_v2)

for assertion in catalog.tags_for_content(content_id):
    source = assertion.provenance
    detail = f" ({source.source_detail})" if source.source_detail else ""
    print(f"{assertion.tag:<8} {source.kind:<6} {source.source_name}@{source.source_version}{detail}")

Both classifier versions remain visible. Archiver does not choose a winner or invent precedence; tag merge and conflict policy remain future work.

## 4. Query by tag without unbounded output

Reverse lookup returns a complete match count plus a bounded tuple of content identities. Add the same user tag to the unique file so the limit becomes visible.

In [ ]:
catalog.add_tag_for_path(root, PurePosixPath("notes/unique.txt"), "favorite", user_source)

result = catalog.search_tagged_content("favorite", limit=1)
print(f"Complete matches: {result.total_matches}")
print(f"Materialized rows: {len(result.contents)}")
for item in result.contents:
    print(f"  {item.content_id.digest[:16]}… ({item.size_bytes} bytes)")

assert result.total_matches == 2
assert len(result.contents) == 1

A provenance filter can narrow the result to content with an active user or system assertion for that tag.

In [ ]:
system_only = catalog.search_tagged_content("binary", provenance="system", limit=20)
assert system_only.total_matches == 1
print("System-derived binary matches:", system_only.total_matches)

## 5. Refreshes and path disappearance do not erase tags

First rename one duplicate and remove the other. After refresh, the renamed path still resolves to the same tagged content.

In [ ]:
renamed = root / "photos" / "renamed.bin"
first.rename(renamed)
second.unlink()
catalog.scan_directory(root)

renamed_tags = catalog.tags_for_path(root, PurePosixPath("photos/renamed.bin"))
assert {item.tag for item in renamed_tags} == {"binary", "favorite"}
print("Tags after rename and refresh:", sorted({item.tag for item in renamed_tags}))

Now remove the final path for the shared bytes. The current scan no longer contains that content, but direct digest lookup still returns its catalog metadata.

In [ ]:
renamed.unlink()
catalog.scan_directory(root)

assert catalog.find_by_content(root, content_id) == []
surviving_tags = catalog.tags_for_content(content_id)
print("Current paths for shared content: 0")
print("Tags retained by digest:", sorted({item.tag for item in surviving_tags}))

## 6. Retraction changes tags, not catalog history

Removing a user tag retracts active user assertions. It leaves system assertions, scan history, observation history, and source files alone.

In [ ]:
scan_history_before = tuple(catalog.scan_history())
observation_history_before = tuple(catalog.observation_history())

retracted = catalog.retract_content_tag(content_id, "favorite")
remaining = catalog.tags_for_content(content_id)

assert retracted == 1
assert {item.tag for item in remaining} == {"binary"}
assert tuple(catalog.scan_history()) == scan_history_before
assert tuple(catalog.observation_history()) == observation_history_before
print("User assertions retracted:", retracted)
print("Remaining producer versions:", [item.provenance.source_version for item in remaining])

## 7. Equivalent CLI workflow

The CLI owns an in-root catalog at `.archiver/catalog.sqlite`. Add/remove commands represent user assertions produced by `archiver-cli`; list/find output includes provenance.

In [ ]:
from archiver.cli import main

cli_root = base / "cli-demo"
cli_root.mkdir()
(cli_root / "example.txt").write_text("CLI example", encoding="utf-8")

assert main(["catalog", "init", str(cli_root)]) == 0
assert main(["catalog", "scan", str(cli_root), "--no-progress"]) == 0
assert main(["catalog", "tags", "add", str(cli_root), "reviewed", "--path", "example.txt"]) == 0
assert main(["catalog", "tags", "list", str(cli_root), "--path", "example.txt"]) == 0
assert main(["catalog", "tags", "find", str(cli_root), "reviewed", "--limit", "5"]) == 0

Existing schema-v1 catalogs are never upgraded merely by opening them. Migration is explicit:

```powershell
uv run archiver catalog migrate C:\path\to\root
```

New catalogs are created directly at schema version 2.

## Exercise

The shared content currently has no path. Add a `reviewed` user assertion directly through its saved `ContentId`, then use reverse lookup to prove it is discoverable.

Try writing the two calls before revealing/running the answer cell.

In [ ]:
# Answer
created = catalog.add_content_tag(content_id, "reviewed", user_source)
reviewed = catalog.search_tagged_content("reviewed", provenance="user", limit=20)

assert created is True
assert reviewed.total_matches == 1
assert reviewed.contents[0].content_id == content_id
print("Reviewed content found by digest:", reviewed.contents[0].content_id.digest[:16] + "…")

## Pitfalls and extensions

- **Do not model path tags.** Resolve a current path to content, or use a known SHA-256 digest when no path remains.
- **Do not label manual work as system-derived.** Provenance should describe the real producer.
- **Do not omit tool versions.** Different versions must remain distinguishable.
- **Do not expect automatic file-kind tags yet.** Extension-based inference is path-dependent; a future classifier should inspect bytes deterministically.
- **Do not expect merge precedence.** Assertions are preserved, but catalog merge/conflict policy is intentionally deferred.

Possible extension: build a byte-based classifier that emits a small deterministic vocabulary and records its tool version plus ruleset identifier in `TagProvenance`.

## Cleanup

In [ ]:
catalog.close()
workspace.cleanup()
print("Temporary catalogs and synthetic files removed.")